# OpenForensics Wild + generated masks — Full VERITAS

**Assigned to:** Member D (4 of 5)
**This notebook trains only:** `full_veritas_ow_v2`

Nine-channel stem + DeepLabV3+. Segmentation loss uses the generated mask on every image.
Segmentation loss uses the generated mask on every image. Dice on Test is agreement with that mask, not a hand-labeled polygon.

Images and labels come from OpenForensics Wild `Real` / `Fake` folders.
The mask for each image is the PNG written by `OW_00_generate_masks.ipynb`.
The crop is the box stored in `index.csv` (the box the mask was painted on).

## Attach

1. New GPU notebook. Do not import this into an OF or WILD training session.
2. Dataset `ahmedgamal/openforensics-wild-dataset`.
3. The finished `OW_00` output (every shard, if `shard_count` was more than 1). The folder must contain `ow_masks/index.csv` and `ow_masks/masks/`.
4. On later epochs: this notebook’s own latest output (`full_veritas_ow_v2_last.pt`).
5. GPU T4, Internet On for the first run (ImageNet B7 download).

Do not attach the Full checkpoint as this run’s weights. Training starts from ImageNet.
Do not attach a different study’s `full_veritas_ow_v2` file; the experiment name is `full_veritas_ow_v2`.

## One epoch per run

Each Save & Run All trains one more epoch and scores Validation. Test runs only after epoch three.
`time_budget_hours` is 11. A mid-epoch checkpoint is written every 30 minutes.
The next run must say `START OF RUN | full_veritas_ow_v2: N/3`.

Set `run_mode` to `full` after `OW_00` has finished every image.

## OW v2 protocol

This is a fresh, separately named experiment. Do not attach legacy or smoke
checkpoints to a full run. Attach the same completed OW_00 masks for every variant.
The B7 backbone and RGB classifier share their initialization procedure across
variants. The late-fusion candidate is a new architecture; corrected early fusion
retains the original layer topology. Loss weights are starting settings, not a
claim of optimality: tune them on Validation, with equal tuning budgets.
Full runs select checkpoints on Validation accuracy and evaluate Test only after
all three epochs. Dice is agreement with teacher-generated masks, not independent GT.


## 1. Environment


In [ ]:
import importlib
import subprocess
import sys

def _pip_install(pkgs):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *pkgs])

missing = []
for mod, pip_name in {
    "cv2": "opencv-python-headless",
    "sklearn": "scikit-learn",
    "pandas": "pandas",
    "tqdm": "tqdm",
}.items():
    try:
        importlib.import_module(mod)
    except Exception:
        missing.append(pip_name)
if missing:
    _pip_install(missing)

import time
import torch
NOTEBOOK_START_TIME = time.time()
print("PyTorch:", torch.__version__, "CUDA:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))


## 2. Imports


In [ ]:
import gc
import hashlib
import json
import math
import os
import random
import shutil
import time
from pathlib import Path

import cv2
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.metrics import accuracy_score, confusion_matrix, f1_score, precision_score, recall_score
from torch.utils.data import DataLoader, Dataset
from torchvision.models import EfficientNet_B7_Weights, efficientnet_b7
from tqdm.auto import tqdm


## 3. Wild images and the generated-mask folder


In [ ]:
def discover_wild_root() -> Path:
    roots = []
    if Path("/kaggle/input").exists():
        roots.append(Path("/kaggle/input"))
    roots.extend([Path.cwd() / "openforensics-wild-dataset", Path.cwd() / "Dataset", Path.cwd()])
    for root in roots:
        if not root.exists():
            continue
        for dirpath, dirnames, _ in os.walk(root):
            p = Path(dirpath)
            if "ow_masks" in p.parts:
                continue
            kids = {d.lower() for d in dirnames}
            if {"train", "validation", "test"} <= kids or {"train", "val", "test"} <= kids:
                return p
            if p.name.lower() == "dataset" and ("train" in kids or "test" in kids):
                return p
    raise FileNotFoundError("Attach ahmedgamal/openforensics-wild-dataset.")


def _split_dir(root: Path, name: str) -> Path:
    aliases = {
        "train": ["Train", "train"],
        "val": ["Validation", "Val", "validation", "val"],
        "test": ["Test", "test"],
    }
    for a in aliases[name]:
        p = root / a
        if p.is_dir():
            return p
    raise FileNotFoundError(f"No {name} split under {root}")


WILD_ROOT = discover_wild_root()
IN_KAGGLE = Path("/kaggle/working").exists()
WORKING = Path("/kaggle/working") if IN_KAGGLE else Path.cwd() / "kaggle_working"
WORKING.mkdir(parents=True, exist_ok=True)
for sub in ("checkpoints", "predictions", "logs"):
    (WORKING / sub).mkdir(parents=True, exist_ok=True)
print("wild root:", WILD_ROOT)
print("working:", WORKING)


## 4. Configuration


In [ ]:
CONFIG = {
    "run_mode": "smoke",
    "seed": 42,
    "implementation_version": "ow_v2",
    "use_pretrained": True,
    "image_size": 600,
    "batch_size": 4,
    "num_workers": 2 if os.name != "nt" else 0,
    "epochs": 3,
    "learning_rate": 1e-4,
    "weight_decay": 1e-4,
    "classification_threshold": 0.5,
    "mask_threshold": 0.5,
    "lambda_cls": 1.0,
    "lambda_seg": 0.25,
    "dice_weight": 1.0,
    "use_amp": True,
    "gradient_accumulation": 1,
    "skip_training": False,
    "resume": True,
    "resume_from_input": True,
    "time_budget_hours": 11.0,
    "epochs_per_run": 1,
    "mid_epoch_checkpoint_minutes": 30,
    "test_eval_when": "final",
    "experiment": "full_veritas_ow_v2",
    "max_per_split": None,
}
EXP_CFG = {
    "multi_stream": True,
    "segmentation": True,
    "legacy_baseline": False,
    "implementation_version": "ow_v2",
    "fusion": "early",
}
if CONFIG["run_mode"] == "smoke":
    CONFIG["max_per_split"] = 64
    CONFIG["epochs"] = 1
    print("SMOKE: up to 64 images per class per split, 1 epoch. Set run_mode='full' after the mask set is complete.")
if CONFIG["run_mode"] == "full" and not torch.cuda.is_available():
    raise RuntimeError("Full training needs GPU T4.")
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("THIS NOTEBOOK TRAINS:", CONFIG["experiment"], EXP_CFG)


## 5. Join each Wild image to its generated mask


In [ ]:
EXTS = {".jpg", ".jpeg", ".png", ".webp", ".bmp"}
LABELS = {"real": 0, "fake": 1}


def rel_key(split: str, class_name: str, file_name: str) -> str:
    return f"{split}/{class_name}/{file_name}"


def _class_dir(split_dir: Path, name: str):
    for a in (name, name.capitalize(), name.upper(), name.lower()):
        p = split_dir / a
        if p.is_dir():
            return p
    return None


def list_images(split_name: str) -> pd.DataFrame:
    split_dir = _split_dir(WILD_ROOT, split_name)
    rows = []
    for cname, lab in LABELS.items():
        cdir = _class_dir(split_dir, cname)
        if cdir is None:
            print("WARNING: missing", split_name, cname)
            continue
        for p in sorted(cdir.iterdir()):
            if p.is_file() and p.suffix.lower() in EXTS:
                rows.append({
                    "path": str(p),
                    "split": split_name,
                    "label": lab,
                    "class_name": cname,
                    "file_name": p.name,
                    "rel_key": rel_key(split_name, cname, p.name),
                })
    return pd.DataFrame(rows)


def unpack_mask_outputs(input_root: Path) -> list:
    import tempfile
    import zipfile
    from pathlib import PurePosixPath

    roots = []
    for archive in input_root.rglob("_output_.zip"):
        with zipfile.ZipFile(archive) as z:
            members = [m for m in z.infolist() if not m.is_dir()
                       and "ow_masks" in PurePosixPath(m.filename.replace("\\", "/")).parts]
            if not members:
                continue
            target = Path(tempfile.mkdtemp(prefix="veritas_ow_masks_"))
            for member in members:
                rel = PurePosixPath(member.filename.replace("\\", "/"))
                if rel.is_absolute() or ".." in rel.parts:
                    raise ValueError(f"Unsafe mask archive path: {member.filename}")
                rel = Path(*rel.parts[rel.parts.index("ow_masks"):])
                dst = target / rel
                if not dst.resolve().is_relative_to(target.resolve()):
                    raise ValueError(f"Unsafe mask archive path: {member.filename}")
                dst.parent.mkdir(parents=True, exist_ok=True)
                with z.open(member) as src, dst.open("wb") as out:
                    shutil.copyfileobj(src, out)
            roots.append(target)
            print("unpacked mask output", archive, "files", len(members), "into", target)
    return roots


def load_mask_table() -> pd.DataFrame:
    frames = []
    roots = []
    for p in (Path("/kaggle/input"), Path("/kaggle/working"), WORKING, Path.cwd()):
        if p.exists():
            roots.append(p)
    roots.extend(unpack_mask_outputs(Path("/kaggle/input")))
    seen = set()
    for root in roots:
        for dirpath, _, filenames in os.walk(root):
            if "index.csv" not in filenames:
                continue
            path = Path(dirpath) / "index.csv"
            key = str(path)
            if key in seen:
                continue
            seen.add(key)
            try:
                df = pd.read_csv(path, dtype=str)
            except Exception as e:
                print("skip", path, e)
                continue
            if not {"image_id", "rel_key", "mask_file", "status", "crop_x0"} <= set(df.columns):
                continue
            df["_root"] = str(path.parent)
            frames.append(df)
            print("mask index", path, "rows", len(df))
    if not frames:
        raise FileNotFoundError(
            "No ow_masks/index.csv found. Run OW_00_generate_masks and attach that notebook output."
        )
    all_df = pd.concat(frames, ignore_index=True)
    kept = []
    used = set()
    for rec in all_df.to_dict("records"):
        iid = str(rec["image_id"])
        if iid in used or str(rec.get("status")) != "ok":
            continue
        png = Path(str(rec["_root"])) / str(rec["mask_file"])
        if not png.is_file():
            continue
        rec["mask_path"] = str(png)
        kept.append(rec)
        used.add(iid)
    if not kept:
        raise FileNotFoundError("index.csv rows were found, but none of the PNG files are present.")
    out = pd.DataFrame(kept).drop_duplicates("rel_key", keep="last")
    print("masks ready", len(out), "teachers", sorted(set(out.teacher.dropna().astype(str))) if "teacher" in out.columns else [])
    return out


MASKS = load_mask_table()
INDEX = {}
for split in ("train", "val", "test"):
    images = list_images(split)
    if images.empty:
        raise RuntimeError(f"{split} has no images")
    merged = images.merge(
        MASKS[["rel_key", "mask_path", "crop_x0", "crop_y0", "crop_x1", "crop_y1", "status", "positive_frac"]],
        on="rel_key",
        how="left",
    )
    have = merged["mask_path"].notna()
    print(f"{split}: images {len(merged)}  with mask {int(have.sum())}  missing {int((~have).sum())}")
    if CONFIG["run_mode"] == "full" and (~have).any():
        missing = merged.loc[~have, "rel_key"].head(8).tolist()
        raise RuntimeError(
            f"{split} is missing {int((~have).sum())} generated masks. Finish OW_00 before full training. Example: {missing}"
        )
    ready = merged.loc[have].copy()
    if CONFIG.get("max_per_split"):
        parts = []
        rng = np.random.RandomState(CONFIG["seed"] + {"train": 1, "val": 2, "test": 3}[split])
        for _, g in ready.groupby("label"):
            n = min(int(CONFIG["max_per_split"]), len(g))
            parts.append(g.sample(n=n, random_state=rng) if n < len(g) else g)
        ready = pd.concat(parts)
    if ready.empty:
        raise RuntimeError(f"{split} has no generated masks. Run OW_00 first.")
    INDEX[split] = ready.reset_index(drop=True)
    print(f"  using {len(INDEX[split])}  fake={int((INDEX[split].label == 1).sum())} real={int((INDEX[split].label == 0).sum())}")


# Hash stable mask metadata, excluding machine-specific mounted paths.
mask_columns = ["rel_key", "label", "mask_file", "crop_x0", "crop_y0", "crop_x1", "crop_y1", "positive_frac", "teacher"]
MASK_SOURCE_HASH = hashlib.sha256(MASKS[mask_columns].sort_values("rel_key").to_csv(index=False).encode()).hexdigest()
_protocol = {k: CONFIG[k] for k in [
    "implementation_version", "run_mode", "seed", "use_pretrained", "image_size",
    "epochs", "batch_size", "learning_rate", "weight_decay", "classification_threshold",
    "mask_threshold", "lambda_cls", "lambda_seg", "dice_weight", "use_amp", "gradient_accumulation",
]}
_protocol.update(mask_source_hash=MASK_SOURCE_HASH, augmentation="aligned_epoch_flip_v2", selection_metric="val_accuracy")
_protocol["data_index_hash"] = hashlib.sha256(pd.concat([
    INDEX[s][["rel_key", "label"]].sort_values("rel_key").assign(split=s)
    for s in ("train", "val", "test")
]).to_csv(index=False).encode()).hexdigest()
PROTOCOL_HASH = hashlib.sha256(json.dumps(_protocol, sort_keys=True).encode()).hexdigest()
print("PROTOCOL_HASH:", PROTOCOL_HASH)
(WORKING / "logs" / "protocol.json").write_text(json.dumps({"hash": PROTOCOL_HASH, "protocol": _protocol,
    "experiment": CONFIG["experiment"], "exp_cfg": EXP_CFG}, indent=2), encoding="utf-8")


## 6. Dataset (saved crop + saved 600 mask)


In [ ]:
IMAGENET_MEAN_NP = np.array([0.485, 0.456, 0.406], dtype=np.float32)
IMAGENET_STD_NP = np.array([0.229, 0.224, 0.225], dtype=np.float32)
IMAGENET_MEAN = torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1)
IMAGENET_STD = torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1)


def letterbox_rgb(image_rgb, size: int):
    h, w = image_rgb.shape[:2]
    scale = min(size / max(h, 1), size / max(w, 1))
    nw, nh = max(int(round(w * scale)), 1), max(int(round(h * scale)), 1)
    resized = cv2.resize(image_rgb, (nw, nh), interpolation=cv2.INTER_LINEAR)
    canvas = np.zeros((size, size, 3), dtype=np.uint8)
    left, top = (size - nw) // 2, (size - nh) // 2
    canvas[top: top + nh, left: left + nw] = resized
    return canvas


class GeneratedMaskDataset(Dataset):
    def __init__(self, df: pd.DataFrame, size: int, augment: bool = False):
        self.df = df.reset_index(drop=True)
        self.size = int(size)
        self.augment = bool(augment)
        self.epoch = 0
        self.seed = int(CONFIG["seed"])

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx: int):
        row = self.df.iloc[idx]
        rng = random.Random(self.seed + self.epoch * len(self.df) + int(idx))
        bgr = cv2.imread(row.path, cv2.IMREAD_COLOR)
        if bgr is None:
            raise FileNotFoundError(f"Unreadable training image: {row.path}")
        else:
            rgb = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)
            h, w = rgb.shape[:2]
            x0 = int(float(row.crop_x0)); y0 = int(float(row.crop_y0))
            x1 = int(float(row.crop_x1)); y1 = int(float(row.crop_y1))
            x0, y0 = max(0, x0), max(0, y0)
            x1, y1 = min(w, max(x0 + 1, x1)), min(h, max(y0 + 1, y1))
        crop = rgb[y0:y1, x0:x1]
        mask = cv2.imread(str(row.mask_path), cv2.IMREAD_GRAYSCALE)
        if mask is None:
            raise FileNotFoundError(f"Unreadable generated mask: {row.mask_path}")
        if mask.shape != (self.size, self.size):
            mask = cv2.resize(mask, (self.size, self.size), interpolation=cv2.INTER_NEAREST)
        if self.augment and rng.random() < 0.5:
            crop = np.clip(crop.astype(np.float32) * (1.0 + rng.uniform(-0.15, 0.15)) + rng.uniform(-12, 12), 0, 255).astype(np.uint8)
        canvas = letterbox_rgb(crop, self.size)
        if self.augment and rng.random() < 0.5:
            canvas = np.ascontiguousarray(canvas[:, ::-1])
            mask = np.ascontiguousarray(mask[:, ::-1])
        x = canvas.astype(np.float32) / 255.0
        x = (x - IMAGENET_MEAN_NP) / IMAGENET_STD_NP
        x = np.transpose(x, (2, 0, 1))
        target = (mask.astype(np.float32) > 127).astype(np.float32)
        return {
            "image": torch.from_numpy(x.copy()),
            "mask": torch.from_numpy(target[None].copy()),
            "label": torch.tensor([float(row.label)], dtype=torch.float32),
            "path": row.path,
            "file_name": row.file_name,
        }


class EpochSeededSampler(torch.utils.data.Sampler):
    def __init__(self, n, seed):
        self.n = int(n)
        self.seed = int(seed)
        self.epoch = 0
        self.start_index = 0

    def set_epoch(self, epoch, start_index=0):
        self.epoch = int(epoch)
        self.start_index = int(start_index)

    def __iter__(self):
        g = torch.Generator()
        g.manual_seed(self.seed * 1000 + self.epoch)
        order = torch.randperm(self.n, generator=g).tolist()
        return iter(order[self.start_index:])

    def __len__(self):
        return max(self.n - self.start_index, 0)


def seed_worker(worker_id):
    worker_seed = torch.initial_seed() % (2**32)
    np.random.seed(worker_seed)
    random.seed(worker_seed)


def make_loader(df, shuffle, augment):
    ds = GeneratedMaskDataset(df, CONFIG["image_size"], augment=augment)
    sampler = EpochSeededSampler(len(ds), CONFIG["seed"]) if shuffle else None
    return DataLoader(
        ds,
        batch_size=CONFIG["batch_size"],
        sampler=sampler,
        shuffle=False,
        num_workers=CONFIG["num_workers"],
        worker_init_fn=seed_worker,
        generator=torch.Generator().manual_seed(CONFIG["seed"]),
        pin_memory=torch.cuda.is_available(),
        drop_last=bool(shuffle and len(ds) >= int(CONFIG["batch_size"])),
    )


if int(CONFIG["image_size"]) != 600:
    raise RuntimeError("Generated masks are 600×600. Leave image_size at 600.")
TRAIN_LOADER = make_loader(INDEX["train"], True, True)
VAL_LOADER = make_loader(INDEX["val"], False, False)
TEST_LOADER = make_loader(INDEX["test"], False, False)
print("train batches", len(TRAIN_LOADER), "val", len(VAL_LOADER), "test", len(TEST_LOADER))


## 7. Model


In [ ]:
def _srm_kernels() -> torch.Tensor:
    k1 = np.array(
        [[0, 0, 0, 0, 0], [0, -1, 2, -1, 0], [0, 2, -4, 2, 0], [0, -1, 2, -1, 0], [0, 0, 0, 0, 0]],
        dtype=np.float32,
    )
    k2 = np.array(
        [[-1, 2, -2, 2, -1], [2, -6, 8, -6, 2], [-2, 8, -12, 8, -2], [2, -6, 8, -6, 2], [-1, 2, -2, 2, -1]],
        dtype=np.float32,
    )
    k3 = np.array(
        [[0, 0, 0, 0, 0], [0, 0, 0, 0, 0], [0, 1, -2, 1, 0], [0, 0, 0, 0, 0], [0, 0, 0, 0, 0]],
        dtype=np.float32,
    )
    ks = []
    for k in (k1, k2, k3):
        s = np.abs(k).sum()
        ks.append(k / (s if s > 0 else 1.0))
    return torch.from_numpy(np.stack(ks, 0)[:, None, :, :])

class ASPP(nn.Module):
    def __init__(self, in_ch: int, out_ch: int = 256, rates=(6, 12, 18)):
        super().__init__()
        self.branches = nn.ModuleList()
        self.branches.append(nn.Sequential(nn.Conv2d(in_ch, out_ch, 1, bias=False), nn.BatchNorm2d(out_ch), nn.ReLU(inplace=True)))
        for r in rates:
            self.branches.append(nn.Sequential(
                nn.Conv2d(in_ch, out_ch, 3, padding=r, dilation=r, bias=False),
                nn.BatchNorm2d(out_ch), nn.ReLU(inplace=True),
            ))
        gn = 32 if out_ch % 32 == 0 else 1
        self.gap = nn.Sequential(
            nn.AdaptiveAvgPool2d(1), nn.Conv2d(in_ch, out_ch, 1, bias=False),
            nn.GroupNorm(gn, out_ch), nn.ReLU(inplace=True),
        )
        self.project = nn.Sequential(
            nn.Conv2d(out_ch * 5, out_ch, 1, bias=False), nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True), nn.Dropout(0.1),
        )

    def forward(self, x):
        res = [b(x) for b in self.branches]
        gap = F.interpolate(self.gap(x), size=x.shape[-2:], mode="bilinear", align_corners=False)
        res.append(gap)
        return self.project(torch.cat(res, dim=1))

class DeepLabV3PlusHead(nn.Module):
    def __init__(self, high_ch: int, low_ch: int, num_classes: int = 1):
        super().__init__()
        self.aspp = ASPP(high_ch, 256)
        self.low_proj = nn.Sequential(nn.Conv2d(low_ch, 48, 1, bias=False), nn.BatchNorm2d(48), nn.ReLU(inplace=True))
        self.decoder = nn.Sequential(
            nn.Conv2d(304, 256, 3, padding=1, bias=False), nn.BatchNorm2d(256), nn.ReLU(inplace=True),
            nn.Conv2d(256, 256, 3, padding=1, bias=False), nn.BatchNorm2d(256), nn.ReLU(inplace=True),
            nn.Conv2d(256, num_classes, 1),
        )

    def forward(self, high, low, out_size):
        high = F.interpolate(self.aspp(high), size=low.shape[-2:], mode="bilinear", align_corners=False)
        x = torch.cat([high, self.low_proj(low)], dim=1)
        return F.interpolate(self.decoder(x), size=out_size, mode="bilinear", align_corners=False)

class ClassificationHead(nn.Module):
    def __init__(self, in_ch: int, dropout: float = 0.5):
        super().__init__()
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.fc = nn.Sequential(nn.Dropout(dropout), nn.Linear(in_ch, 1))

    def forward(self, feat):
        return self.fc(self.pool(feat).flatten(1))

"""Versioned OW models: corrected early fusion and an experimental late fusion.

Legacy checkpoints continue to use architecture.py. These models require fresh
training and record implementation_version='ow_v2' in their checkpoints.
"""

import torch

from torch import nn

from torch.nn import functional as F

from torchvision.models import EfficientNet_B7_Weights, efficientnet_b7

IMPLEMENTATION_VERSION = "ow_v2"

class ForensicStreams(nn.Module):
    def __init__(self):
        super().__init__()
        self.register_buffer("srm_weight", _srm_kernels())

    def forward(self, normalized_rgb):
        # FFT on 600px images must stay float32 even under CUDA autocast.
        with torch.autocast(device_type=normalized_rgb.device.type, enabled=False):
            x = normalized_rgb.float()
            mean = IMAGENET_MEAN.to(x.device)
            std = IMAGENET_STD.to(x.device)
            rgb = (x * std + mean).clamp(0, 1)
            gray = (rgb * x.new_tensor([0.299, 0.587, 0.114]).view(1, 3, 1, 1)).sum(1, keepdim=True)
            residual = F.conv2d(gray, self.srm_weight.float(), padding=2).tanh()
            spectrum = torch.fft.fftshift(torch.fft.fft2(rgb, norm="ortho"), dim=(-2, -1))
            magnitude = torch.log1p(spectrum.abs())
            lo = magnitude.amin(dim=(-2, -1), keepdim=True)
            span = magnitude.amax(dim=(-2, -1), keepdim=True) - lo
            magnitude = (magnitude - lo) / span.clamp_min(1e-6)
            magnitude = magnitude - magnitude.mean(dim=(-2, -1), keepdim=True)
        return residual, magnitude

def expand_rgb_stem(conv3):
    """Preserve the RGB response exactly; learn the auxiliary channels from zero."""
    conv9 = nn.Conv2d(9, conv3.out_channels, conv3.kernel_size, stride=conv3.stride,
                      padding=conv3.padding, dilation=conv3.dilation,
                      bias=conv3.bias is not None, device=conv3.weight.device, dtype=conv3.weight.dtype)
    with torch.no_grad():
        conv9.weight.zero_()
        conv9.weight[:, :3].copy_(conv3.weight)
        if conv3.bias is not None:
            conv9.bias.copy_(conv3.bias)
    return conv9

class AuxiliaryEncoder(nn.Sequential):
    def __init__(self):
        layers = []
        channels = 3
        for width in (32, 64, 128):
            layers.extend([nn.Conv2d(channels, width, 3, stride=2, padding=1, bias=False),
                           nn.GroupNorm(8, width), nn.SiLU()])
            channels = width
        layers.extend([nn.AdaptiveAvgPool2d(1), nn.Flatten(1)])
        super().__init__(*layers)

class VERITASV2(nn.Module):
    def __init__(self, multi_stream=True, segmentation=True, pretrained=True, fusion="early"):
        super().__init__()
        if fusion not in ("rgb", "early", "late") or multi_stream != (fusion != "rgb"):
            raise ValueError("Use fusion='rgb' for RGB-only models, otherwise 'early' or 'late'")
        self.multi_stream = bool(multi_stream)
        self.segmentation = bool(segmentation)
        self.fusion = fusion
        weights = EfficientNet_B7_Weights.IMAGENET1K_V1 if pretrained else None
        self.encoder = efficientnet_b7(weights=weights).features
        # Create the common RGB backbone/head before variant-specific parameters.
        self.cls_head = ClassificationHead(2560)
        self.seg_head = DeepLabV3PlusHead(224, 48, 1) if segmentation else None
        self.streams = ForensicStreams() if multi_stream else None
        if fusion == "early":
            self.encoder[0][0] = expand_rgb_stem(self.encoder[0][0])
        elif fusion == "late":
            self.srm_encoder = AuxiliaryEncoder()
            self.fft_encoder = AuxiliaryEncoder()
            self.aux_projection = nn.Linear(256, 2560, bias=False)
            nn.init.zeros_(self.aux_projection.weight)

    def forward(self, x):
        size = x.shape[-2:]
        residual = magnitude = None
        if self.multi_stream:
            residual, magnitude = self.streams(x)
        if self.fusion == "early":
            # Keep the original ImageNet-normalized RGB, unlike the legacy stem.
            x = torch.cat([x, residual, magnitude], dim=1)
        low = high = None
        for i, stage in enumerate(self.encoder):
            x = stage(x)
            if i == 2:
                low = x
            elif i == 5:
                high = x
        if self.fusion == "late":
            # Pool independent domains before fusion: frequency coordinates are
            # never treated as image locations by the segmentation decoder.
            auxiliary = torch.cat([self.srm_encoder(residual), self.fft_encoder(magnitude)], dim=1)
            pooled = self.cls_head.pool(x).flatten(1) + self.aux_projection(auxiliary)
            logits = self.cls_head.fc(pooled)
        else:
            logits = self.cls_head(x)
        out = {"classification_logits": logits, "fake_probability": torch.sigmoid(logits)}
        if self.segmentation:
            seg_logits = self.seg_head(high, low, out_size=size)
            out.update(segmentation_logits=seg_logits, segmentation_prob=torch.sigmoid(seg_logits))
        return out

def build_model():
    return VERITASV2(multi_stream=EXP_CFG["multi_stream"], segmentation=EXP_CFG["segmentation"],
                     pretrained=CONFIG["use_pretrained"], fusion=EXP_CFG["fusion"]).to(DEVICE)


In [ ]:
def set_seed(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(CONFIG["seed"])
print("legacy_baseline", EXP_CFG["legacy_baseline"], "multi_stream", EXP_CFG["multi_stream"], "segmentation", EXP_CFG["segmentation"])


## 8. Loss


In [ ]:
"""Loss and checkpoint safeguards shared by the versioned OW notebooks."""
import torch
from torch import nn


class VERITASLossV2(nn.Module):
    def __init__(self, lambda_cls=1.0, lambda_seg=0.25, dice_weight=1.0, segmentation=False):
        super().__init__()
        if lambda_cls <= 0 or lambda_seg < 0 or dice_weight < 0:
            raise ValueError("Loss weights must be nonnegative, with positive classification weight")
        self.lambda_cls = float(lambda_cls)
        self.lambda_seg = float(lambda_seg)
        self.dice_weight = float(dice_weight)
        self.segmentation = bool(segmentation)
        self.bce = nn.BCEWithLogitsLoss()

    def forward(self, outputs, labels, masks=None):
        cls = self.bce(outputs["classification_logits"], labels)
        seg_bce = seg_dice = cls.new_zeros(())
        if self.segmentation:
            if masks is None:
                raise ValueError("Segmentation targets are required")
            logits = outputs["segmentation_logits"]
            seg_bce = self.bce(logits, masks)
            # 600x600 reductions exceed float16's range under CUDA autocast.
            probs = logits.float().sigmoid()
            masks = masks.float()
            dims = (1, 2, 3)
            intersection = (probs * masks).sum(dims)
            denominator = probs.sum(dims) + masks.sum(dims)
            seg_dice = 1 - ((2 * intersection + 1e-6) / (denominator + 1e-6)).mean()
        # Average component losses; lambda_seg then controls the auxiliary task.
        seg = (seg_bce + self.dice_weight * seg_dice) / (1 + self.dice_weight)
        total = self.lambda_cls * cls + self.lambda_seg * seg
        return dict(total=total, classification=cls.detach(), segmentation=seg.detach(),
                    segmentation_bce=seg_bce.detach(), segmentation_dice=seg_dice.detach())


def validate_resume_checkpoint(checkpoint, experiment, exp_cfg, protocol_hash):
    if checkpoint.get("experiment_name") != experiment:
        raise ValueError("Checkpoint belongs to a different experiment")
    if checkpoint.get("exp_cfg") != exp_cfg:
        raise ValueError("Checkpoint architecture/version differs; start a fresh experiment")
    if checkpoint.get("protocol_hash") != protocol_hash:
        raise ValueError("Checkpoint protocol differs (including smoke/full or mask source)")


## 9. Checkpoint resume (one epoch per run)


In [ ]:
EXP = CONFIG["experiment"]


def _torch_load(path):
    try:
        return torch.load(path, map_location="cpu", weights_only=False)
    except TypeError:
        return torch.load(path, map_location="cpu")


def checkpoint_progress(path: Path):
    try:
        ck = _torch_load(path)
        partial = ck.get("partial") or {}
        return (int(ck.get("epoch") or 0), int(partial.get("step", 0)))
    except Exception as e:
        print("could not read", path, e)
        return (-1, -1)


def stage_checkpoints_from_input():
    if not CONFIG.get("resume_from_input", True):
        return
    ckpt_dir = WORKING / "checkpoints"
    names = {f"{EXP}_last.pt": [], f"{EXP}_best.pt": []}
    root = Path("/kaggle/input")
    if not root.exists():
        return
    for dirpath, _, filenames in os.walk(root):
        for fn in filenames:
            if fn in names:
                names[fn].append(Path(dirpath) / fn)
    for fn, paths in names.items():
        if not paths:
            continue
        dst = ckpt_dir / fn
        pool = ([dst] if dst.exists() else []) + list(paths)
        best = max(pool, key=lambda p: checkpoint_progress(p))
        if best != dst:
            shutil.copy2(best, dst)
            print("staged", best, "->", dst, checkpoint_progress(best))


def save_checkpoint(path, model, optimizer, scaler, epoch, best_metric, history, epoch_seconds, partial=None):
    path = Path(path)
    tmp = path.with_name(path.name + ".tmp")
    torch.save(
        {
            "protocol_hash": PROTOCOL_HASH,
            "selection_metric": "val_accuracy",
            "torch_rng_state": torch.get_rng_state(),
            "cuda_rng_state": torch.cuda.get_rng_state() if DEVICE.type == "cuda" else None,
            "epoch": epoch,
            "partial": partial,
            "model_state_dict": model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict() if optimizer is not None else None,
            "scaler_state_dict": scaler.state_dict() if scaler is not None else None,
            "best_metric": best_metric,
            "experiment_name": EXP,
            "history": history,
            "epoch_seconds": epoch_seconds,
            "config": CONFIG,
            "exp_cfg": EXP_CFG,
        },
        tmp,
    )
    tmp.replace(path)


def load_if_any(path, model, optimizer=None):
    path = Path(path)
    if not path.exists():
        return 0, -1.0, {}
    ck = _torch_load(path)
    validate_resume_checkpoint(ck, EXP, EXP_CFG, PROTOCOL_HASH)
    model.load_state_dict(ck["model_state_dict"])
    if optimizer is not None and ck.get("optimizer_state_dict"):
        try:
            optimizer.load_state_dict(ck["optimizer_state_dict"])
        except Exception as e:
            print("optimizer state not loaded:", e)
    start = int(ck.get("epoch") or 0)
    best = float(ck["best_metric"]) if ck.get("best_metric") is not None else -1.0
    print("resumed", path, "epochs_completed", start, "best", best)
    return start, best, ck


## 10. Train and test


In [ ]:
def mask_metrics_from_tensors(probs, targets, thresh=0.5):
    probs = probs.numpy() if torch.is_tensor(probs) else np.asarray(probs)
    targets = targets.numpy() if torch.is_tensor(targets) else np.asarray(targets)
    ious, dices = [], []
    for i in range(len(targets)):
        p = (probs[i].reshape(-1) >= thresh).astype(np.float32)
        t = (targets[i].reshape(-1) >= 0.5).astype(np.float32)
        inter = float((p * t).sum())
        union = float(((p + t) > 0).sum())
        ps, ts = float(p.sum()), float(t.sum())
        ious.append(1.0 if union == 0 else inter / union)
        dices.append(1.0 if (ps + ts) == 0 else (2 * inter) / (ps + ts))
    return ious, dices


@torch.no_grad()
def evaluate_loader(model, loader, split_name: str):
    model.eval()
    y_true, y_pred, y_prob, recs = [], [], [], []
    dice_all, iou_all = [], []
    use_seg = bool(getattr(model, "segmentation", False))
    for batch in loader:
        images = batch["image"].to(DEVICE, non_blocking=True)
        labels = batch["label"].to(DEVICE, non_blocking=True)
        masks = batch["mask"].to(DEVICE, non_blocking=True)
        out = model(images)
        prob = out["fake_probability"].detach().cpu().numpy().reshape(-1)
        pred = (prob >= CONFIG["classification_threshold"]).astype(int)
        true = labels.detach().cpu().numpy().reshape(-1).astype(int)
        y_true.extend(true.tolist()); y_pred.extend(pred.tolist()); y_prob.extend(prob.tolist())
        mm = None
        if use_seg and "segmentation_prob" in out:
            mm = mask_metrics_from_tensors(out["segmentation_prob"].detach().cpu(), masks.detach().cpu(), CONFIG["mask_threshold"])
            iou_all.extend(mm[0]); dice_all.extend(mm[1])
        for i in range(len(true)):
            rec = {
                "file_name": batch["file_name"][i],
                "path": batch["path"][i],
                "split": split_name,
                "ground_truth_label": int(true[i]),
                "predicted_label": int(pred[i]),
                "fake_probability": float(prob[i]),
            }
            if mm is not None:
                rec["iou"] = mm[0][i]
                rec["dice"] = mm[1][i]
            recs.append(rec)
    metrics = {
        "accuracy": float(accuracy_score(y_true, y_pred)) if y_true else float("nan"),
        "f1": float(f1_score(y_true, y_pred, zero_division=0)) if y_true else float("nan"),
        "precision": float(precision_score(y_true, y_pred, zero_division=0)) if y_true else float("nan"),
        "recall": float(recall_score(y_true, y_pred, zero_division=0)) if y_true else float("nan"),
        "n": int(len(y_true)),
    }
    if y_true:
        tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
        metrics.update({"tp": int(tp), "tn": int(tn), "fp": int(fp), "fn": int(fn)})
    if use_seg and dice_all:
        metrics["dice"] = float(np.mean(dice_all))
        metrics["miou"] = float(np.mean(iou_all))
        metrics["dice_note"] = "agreement with the generated mask, not a hand-labeled polygon"
    return metrics, recs


def train_one_experiment():
    model = build_model()
    use_seg = bool(getattr(model, "segmentation", False))
    criterion = VERITASLossV2(CONFIG["lambda_cls"], CONFIG["lambda_seg"], CONFIG["dice_weight"], use_seg)
    optimizer = torch.optim.AdamW(model.parameters(), lr=CONFIG["learning_rate"], weight_decay=CONFIG["weight_decay"])
    total_epochs = max(int(CONFIG["epochs"]), 1)
    scheduler = torch.optim.lr_scheduler.LambdaLR(
        optimizer, lambda e: 0.5 * (1.0 + math.cos(math.pi * min(e, total_epochs) / total_epochs))
    )
    stage_checkpoints_from_input()
    ckpt_best = WORKING / "checkpoints" / f"{EXP}_best.pt"
    ckpt_last = WORKING / "checkpoints" / f"{EXP}_last.pt"
    resume_path = ckpt_last if ckpt_last.exists() else ckpt_best
    start_epoch, best_metric, resumed = 0, -1.0, {}
    if CONFIG.get("resume", True):
        start_epoch, best_metric, resumed = load_if_any(resume_path, model, optimizer)
    for _ in range(start_epoch):
        scheduler.step()
    history = list((resumed or {}).get("history", []))
    epoch_seconds = list((resumed or {}).get("epoch_seconds", []))
    partial = (resumed or {}).get("partial") if resume_path == ckpt_last else None
    best_metric = -1.0 if best_metric is None else float(best_metric)
    amp_ok = bool(CONFIG["use_amp"] and DEVICE.type == "cuda")
    try:
        scaler = torch.amp.GradScaler("cuda", enabled=amp_ok)
        autocast = lambda: torch.amp.autocast("cuda", enabled=amp_ok)
    except Exception:
        scaler = torch.cuda.amp.GradScaler(enabled=amp_ok)
        autocast = lambda: torch.cuda.amp.autocast(enabled=amp_ok)
    scaler_state = (resumed or {}).get("scaler_state_dict")
    if scaler_state and amp_ok:
        try:
            scaler.load_state_dict(scaler_state)
        except Exception as e:
            print("scaler not loaded:", e)
    per_run = CONFIG.get("epochs_per_run")
    stop_after = total_epochs if not per_run else min(total_epochs, start_epoch + int(per_run))
    print(f"START OF RUN | {EXP}: {start_epoch}/{total_epochs} epochs completed")
    if CONFIG["skip_training"]:
        if ckpt_best.exists():
            load_if_any(ckpt_best, model)
        return model, start_epoch, best_metric, history
    budget_s = float(CONFIG.get("time_budget_hours") or 11) * 3600.0
    elapsed = lambda: time.time() - NOTEBOOK_START_TIME
    accum = max(int(CONFIG["gradient_accumulation"]), 1)
    mid_s = float(CONFIG.get("mid_epoch_checkpoint_minutes") or 30) * 60.0
    sampler = getattr(TRAIN_LOADER, "sampler", None)
    for epoch in range(start_epoch, stop_after):
        start_step = 0
        if partial is not None and int(partial.get("epoch", -1)) == epoch and isinstance(sampler, EpochSeededSampler):
            start_step = int(partial["step"])
            print(f"Continuing epoch {epoch + 1} from step {start_step}/{len(TRAIN_LOADER)}")
        epoch_partial = partial if start_step else None
        partial = None
        if isinstance(sampler, EpochSeededSampler):
            sampler.set_epoch(epoch, start_step * int(CONFIG["batch_size"]))
        set_seed(CONFIG["seed"] + epoch)
        TRAIN_LOADER.dataset.epoch = epoch
        TRAIN_LOADER.generator.manual_seed(CONFIG["seed"] + epoch)
        if start_step and resumed.get("torch_rng_state") is not None:
            torch.set_rng_state(resumed["torch_rng_state"])
            if DEVICE.type == "cuda" and resumed.get("cuda_rng_state") is not None:
                torch.cuda.set_rng_state(resumed["cuda_rng_state"], DEVICE)
        total_steps = start_step + len(TRAIN_LOADER)
        model.train()
        optimizer.zero_grad(set_to_none=True)
        running_cls = running_seg_bce = running_seg_dice = 0.0
        running, seen, t0, last_save = 0.0, 0, time.time(), time.time()
        if epoch_partial:
            running, running_cls, running_seg_bce, running_seg_dice = epoch_partial["loss_sums"]
            seen = int(epoch_partial["seen"])
            t0 -= float(epoch_partial["elapsed_seconds"])
        it = tqdm(TRAIN_LOADER, desc=f"{EXP} epoch {epoch + 1}", leave=True)
        for i, batch in enumerate(it):
            step = start_step + i
            images = batch["image"].to(DEVICE, non_blocking=True)
            labels = batch["label"].to(DEVICE, non_blocking=True)
            masks = batch["mask"].to(DEVICE, non_blocking=True)
            with autocast():
                out = model(images)
                losses = criterion(out, labels, masks if use_seg else None)
                loss = losses["total"] / min(accum, total_steps - (step // accum) * accum)
            scaler.scale(loss).backward()
            update_boundary = (step + 1) % accum == 0 or step + 1 == total_steps
            if update_boundary:
                scaler.step(optimizer)
                scaler.update()
                optimizer.zero_grad(set_to_none=True)
            running += float(losses["total"].detach()) * images.size(0)
            running_cls += float(losses["classification"]) * images.size(0)
            running_seg_bce += float(losses["segmentation_bce"]) * images.size(0)
            running_seg_dice += float(losses["segmentation_dice"]) * images.size(0)
            seen += images.size(0)
            if (i + 1) % 10 == 0:
                it.set_postfix(loss=f"{float(loss.item()) * accum:.4f}", step=step + 1)
            if update_boundary and (time.time() - last_save) >= mid_s:
                save_checkpoint(ckpt_last, model, optimizer, scaler, epoch, best_metric, history, epoch_seconds, partial={"epoch": epoch, "step": step + 1,
                    "loss_sums": [running, running_cls, running_seg_bce, running_seg_dice],
                    "seen": seen, "elapsed_seconds": time.time() - t0})
                last_save = time.time()
                print(f"mid-epoch save epoch {epoch + 1} step {step + 1}/{len(TRAIN_LOADER)}")
            if update_boundary and elapsed() > budget_s:
                save_checkpoint(ckpt_last, model, optimizer, scaler, epoch, best_metric, history, epoch_seconds, partial={"epoch": epoch, "step": step + 1,
                    "loss_sums": [running, running_cls, running_seg_bce, running_seg_dice],
                    "seen": seen, "elapsed_seconds": time.time() - t0})
                print("time budget hit; saved mid-epoch and stopping")
                (WORKING / "logs" / f"{EXP}_training_progress.json").write_text(
                    json.dumps({"experiment": EXP, "epochs_completed": epoch, "epochs_total": total_epochs, "partial_step": step + 1}, indent=2),
                    encoding="utf-8",
                )
                return model, epoch, best_metric, history
            del images, labels, masks, out, loss
        scheduler.step()
        train_loss = running / max(seen, 1)
        val_metrics, _ = evaluate_loader(model, VAL_LOADER, "val")
        hist = {"epoch": epoch + 1, "train_loss": train_loss, **{f"val_{k}": v for k, v in val_metrics.items()}}
        hist.update(train_classification=running_cls / max(seen, 1),
                    train_segmentation_bce=running_seg_bce / max(seen, 1),
                    train_segmentation_dice=running_seg_dice / max(seen, 1), lambda_seg=CONFIG["lambda_seg"])
        history.append(hist)
        epoch_seconds.append(time.time() - t0)
        print("END OF EPOCH", hist)
        score = float(val_metrics["accuracy"])
        if score >= best_metric:
            best_metric = score
            save_checkpoint(ckpt_best, model, optimizer, scaler, epoch + 1, best_metric, history, epoch_seconds)
            print("new best val accuracy", best_metric)
        save_checkpoint(ckpt_last, model, optimizer, scaler, epoch + 1, best_metric, history, epoch_seconds)
        (WORKING / "logs" / f"{EXP}_training_progress.json").write_text(
            json.dumps({"experiment": EXP, "epochs_completed": epoch + 1, "epochs_total": total_epochs, "best_val_accuracy": best_metric}, indent=2),
            encoding="utf-8",
        )
        print(f"END OF RUN | {EXP}: {epoch + 1}/{total_epochs} epochs completed")
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
    return model, stop_after, best_metric, history


MODEL, EPOCHS_DONE, BEST, HISTORY = train_one_experiment()
print("epochs done", EPOCHS_DONE, "best val accuracy", BEST)
over_budget = (time.time() - NOTEBOOK_START_TIME) > float(CONFIG["time_budget_hours"]) * 3600.0
if over_budget:
    print("time budget already used; skipping Test this run")
elif EPOCHS_DONE >= int(CONFIG["epochs"]) or CONFIG.get("test_eval_when") == "always":
    ckpt_best = WORKING / "checkpoints" / f"{EXP}_best.pt"
    if ckpt_best.exists():
        load_if_any(ckpt_best, MODEL)
    metrics, recs = evaluate_loader(MODEL, TEST_LOADER, "test")
    print("TEST", metrics)
    pred_path = WORKING / "predictions" / f"{EXP}_predictions.csv"
    pd.DataFrame(recs).to_csv(pred_path, index=False)
    row = {"experiment": EXP, "status": "ok", "epochs_completed": EPOCHS_DONE, "split": "test", "protocol_hash": PROTOCOL_HASH, "implementation_version": "ow_v2", **{k: v for k, v in metrics.items() if not str(k).endswith("note")}}
    pd.DataFrame([row]).to_csv(WORKING / "experiment_results.csv", index=False)
    print("wrote", pred_path)


if HISTORY:
    pd.DataFrame(HISTORY).to_csv(WORKING / "logs" / "validation_history.csv", index=False)
print("RUN_STATE", json.dumps({"experiment": EXP, "epochs_completed": EPOCHS_DONE,
    "epochs_total": CONFIG["epochs"], "protocol_hash": PROTOCOL_HASH, "exp_cfg": EXP_CFG}))
if CONFIG["run_mode"] == "smoke":
    assert EPOCHS_DONE == 1 and len(INDEX["test"]) == metrics["n"] == 128
    assert len(MASKS) == 190335
    assert all(math.isfinite(metrics[k]) for k in ("accuracy", "f1", "precision", "recall"))
    assert (WORKING / "checkpoints" / f"{EXP}_last.pt").is_file()
    print("TRAIN_SMOKE_VALIDATION_PASSED", EXP)


## Notes

- `full_veritas_ow_v2` starts from ImageNet. It does not resume `full_veritas_defacto`.
- Classification labels are the Wild folder names. Real = 0, fake = 1.
- The mask PNG is the current Full model’s prediction on that crop. Dice measures agreement with that prediction.
- All five `OWV2_01`–`OWV2_05` notebooks must print the same `PROTOCOL_HASH` when `run_mode` matches.
- `OWV2_06_compare.ipynb` reads the `experiment_results.csv` files. It does not train.
